# Ejercicio 5 — Pipeline de scikit-learn con MLflow en Databricks

Este notebook adapta un pipeline de **scikit-learn** para usar tres funcionalidades de **MLflow** en Databricks:

1. **Experiments**: cada modelo candidato se entrena en un *run* donde se registran parámetros, métricas y artefactos.
2. **Models**: el pipeline completo (preprocesamiento + modelo) se guarda como MLflow Model, con firma y ejemplo de entrada.
3. **Model Registry en Unity Catalog**: el mejor modelo se registra, recibe el alias `champion` y se carga desde el registro para predecir.

**Problema:** clasificar tumores como malignos o benignos (dataset *Breast Cancer Wisconsin*, incluido en scikit-learn).

## 0. Instalación y configuración

In [ ]:
%pip install -q "mlflow>=3" scikit-learn matplotlib
dbutils.library.restartPython()

In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import matplotlib.pyplot as plt

from mlflow.models import infer_signature
from mlflow import MlflowClient
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, ConfusionMatrixDisplay)

print("MLflow", mlflow.__version__)

In [ ]:
# Experimento en la carpeta del usuario actual del workspace
usuario = spark.sql("SELECT current_user()").first()[0]
EXPERIMENTO = f"/Users/{usuario}/ejercicio5_mlflow_sklearn"
mlflow.set_experiment(EXPERIMENTO)

# Registro de modelos en Unity Catalog (catálogo.esquema.modelo)
mlflow.set_registry_uri("databricks-uc")
MODELO_UC = "workspace.default.ejercicio5_cancer_clf"

print("Experimento:", EXPERIMENTO)
print("Modelo UC:  ", MODELO_UC)

## 1. Datos

In [ ]:
datos = load_breast_cancer(as_frame=True)
X, y = datos.data, datos.target          # 0 = maligno, 1 = benigno

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)
y.value_counts().rename({0: "maligno", 1: "benigno"})

## 2. Pipeline de scikit-learn

El preprocesamiento y el modelo viven dentro del mismo `Pipeline`. Así MLflow guarda **todo junto** y no hay diferencias entre entrenamiento e inferencia.

In [ ]:
def crear_pipeline(modelo):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("modelo", modelo),
    ])

candidatos = {
    "logistic_regression": LogisticRegression(C=1.0, max_iter=1000),
    "random_forest": RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42),
    "gradient_boosting": GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, random_state=42),
}

## 3. Experiments: entrenar y registrar cada candidato

Cada modelo es un *run* hijo dentro de un *run* padre. En cada uno se registran:
- **Parámetros** del modelo y del pipeline
- **Métricas** de validación cruzada y de prueba
- **Artefactos**: matriz de confusión
- **Modelo**: el pipeline completo, con firma (`signature`) y ejemplo de entrada

In [ ]:
def evaluar(pipe, X, y):
    pred = pipe.predict(X)
    prob = pipe.predict_proba(X)[:, 1]
    return {
        "test_accuracy": accuracy_score(y, pred),
        "test_precision": precision_score(y, pred),
        "test_recall": recall_score(y, pred),
        "test_f1": f1_score(y, pred),
        "test_roc_auc": roc_auc_score(y, prob),
    }

resultados = []

with mlflow.start_run(run_name="comparacion_modelos") as run_padre:
    mlflow.set_tags({"proyecto": "ejercicio5", "dataset": "breast_cancer"})
    mlflow.log_input(mlflow.data.from_pandas(X_train.assign(target=y_train), targets="target",
                                             name="breast_cancer_train"), context="training")

    for nombre, modelo in candidatos.items():
        with mlflow.start_run(run_name=nombre, nested=True) as run:
            pipe = crear_pipeline(modelo)

            # Validación cruzada sobre entrenamiento
            cv_f1 = cross_val_score(pipe, X_train, y_train, cv=5, scoring="f1")
            pipe.fit(X_train, y_train)
            metricas = evaluar(pipe, X_test, y_test)
            metricas.update({"cv_f1_mean": cv_f1.mean(), "cv_f1_std": cv_f1.std()})

            # Parámetros y métricas
            mlflow.log_param("algoritmo", nombre)
            mlflow.log_params({k: v for k, v in modelo.get_params().items()
                               if isinstance(v, (int, float, str, bool)) or v is None})
            mlflow.log_metrics(metricas)

            # Artefacto: matriz de confusión
            fig, ax = plt.subplots(figsize=(4, 4))
            ConfusionMatrixDisplay.from_estimator(pipe, X_test, y_test,
                                                  display_labels=["maligno", "benigno"], ax=ax)
            ax.set_title(nombre)
            mlflow.log_figure(fig, "confusion_matrix.png")
            plt.close(fig)

            # Modelo: pipeline completo con firma
            firma = infer_signature(X_train, pipe.predict(X_train))
            info = mlflow.sklearn.log_model(
                sk_model=pipe,
                name="modelo",
                signature=firma,
                input_example=X_train.head(3),
                serialization_format="cloudpickle",
            )

            resultados.append({"modelo": nombre, "run_id": run.info.run_id,
                               "model_uri": info.model_uri, **metricas})

tabla = pd.DataFrame(resultados).sort_values("test_f1", ascending=False)
tabla[["modelo", "cv_f1_mean", "test_accuracy", "test_f1", "test_roc_auc"]].round(4)

## 4. Buscar el mejor modelo con la API de MLflow

Los *runs* se consultan como un DataFrame con `mlflow.search_runs`, igual que en la pestaña **Experiments** de Databricks.

In [ ]:
runs = mlflow.search_runs(
    filter_string=f"tags.mlflow.parentRunId = '{run_padre.info.run_id}'",
    order_by=["metrics.test_f1 DESC", "metrics.test_roc_auc DESC"],
)
mejor = runs.iloc[0]
print("Mejor modelo:", mejor["params.algoritmo"],
      "| F1 =", round(mejor["metrics.test_f1"], 4),
      "| AUC =", round(mejor["metrics.test_roc_auc"], 4))

mejor_uri = tabla.set_index("run_id").loc[mejor["run_id"], "model_uri"]
mejor_uri

## 5. Models: registrar el mejor en Unity Catalog

Se crea una nueva versión del modelo en `workspace.default.ejercicio5_cancer_clf` y se le asigna el alias **`champion`**.

In [ ]:
version = mlflow.register_model(model_uri=mejor_uri, name=MODELO_UC)

cliente = MlflowClient()
cliente.set_registered_model_alias(MODELO_UC, "champion", version.version)
cliente.update_model_version(
    MODELO_UC, version.version,
    description=f"{mejor['params.algoritmo']} | test F1 = {mejor['metrics.test_f1']:.4f}",
)
print(f"Registrado {MODELO_UC} v{version.version} con alias 'champion'")

## 6. Cargar el modelo desde el registro y predecir

Cualquier proceso (otro notebook, un job o Model Serving) puede usar el modelo por su alias, sin saber qué *run* lo produjo.

In [ ]:
modelo_prod = mlflow.pyfunc.load_model(f"models:/{MODELO_UC}@champion")

muestra = X_test.head(10)
pd.DataFrame({
    "real": y_test.head(10).map({0: "maligno", 1: "benigno"}).values,
    "prediccion": pd.Series(modelo_prod.predict(muestra)).map({0: "maligno", 1: "benigno"}).values,
})

## Resumen

| Funcionalidad de MLflow | Dónde se usa |
|---|---|
| **Experiments** | `set_experiment`, *runs* padre/hijo, `log_params`, `log_metrics`, `log_figure`, `log_input` |
| **Models** | `mlflow.sklearn.log_model` con `signature` e `input_example` |
| **Model Registry (Unity Catalog)** | `register_model`, alias `champion`, carga con `models:/...@champion` |

Para ver los resultados: menú **Experiments** → `ejercicio5_mlflow_sklearn`, y **Catalog** → `workspace` → `default` → *Models*.